# 20 — Croisement de zonages et occupation du sol

**Objectif :** produire un tableau croisé de surfaces et vérifier la couverture,
les lacunes et les recouvrements. Les données de ce cours sont entièrement
**simulées** : leurs rectangles permettent de calculer les résultats à la main.
Ils ne décrivent aucun territoire ou inventaire forestier réel.

Le même moteur sert à Python, au terminal, aux plans et à
**Vecteur → Table croisée des surfaces**. [Contrat G09](../../docs/VECTOR_TABULATION.md).

In [1]:
import hashlib
import json
from pathlib import Path
import subprocess
import sys
from threading import Event
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import box
import cartomize as cm
from cartomize.vector_io import write_geopackage
from cartomize.viewer_data import table_page
from course import run_directory

work = run_directory('20_surfaces')
assert tuple(map(int, cm.__version__.split('.')[:2])) >= (1, 11)
print('Cartomize', cm.__version__)

Cartomize 1.11.0


## 1. Deux zones de 1 hectare, des catégories et des doublons

Chaque zone mesure 100 × 100 m, soit 1 ha. Une partie de la zone `001` est
présente deux fois dans la source. Une partie de la forêt est également
dupliquée. L'eau traverse les deux zones ; une catégorie extérieure permet
de vérifier que les cellules de surface nulle restent dans la table.

EPSG:32733 sert ici de repère métrique aux rectangles simulés. Pour une étude,
choisir une projection adaptée à l'emprise et à la mesure de surface.

In [2]:
x0, y0 = 500_000, 9_500_000
def rectangle(left, bottom, right, top):
    return box(x0 + left, y0 + bottom, x0 + right, y0 + top)

zones = gpd.GeoDataFrame({'territoire': ['001', 'NA', '001']}, geometry=[
    rectangle(0, 0, 100, 100), rectangle(100, 0, 200, 100),
    rectangle(0, 0, 50, 100)], crs=32733)
classes = gpd.GeoDataFrame({'occupation': ['Forêt', 'Eau', 'Forêt', 'Agriculture']}, geometry=[
    rectangle(0, 0, 60, 100), rectangle(60, 0, 150, 100),
    rectangle(0, 0, 20, 100), rectangle(300, 0, 310, 100)], crs=32733)
original_wkb = [zones.geometry.to_wkb().tolist(), classes.geometry.to_wkb().tolist()]
parameters = dict(zone_field='territoire', class_field='occupation', area_unit='ha')
display(zones.drop(columns='geometry'))

,territoire
0,001
1,NA
2,001


## 2. Table détaillée et matrice

La forêt doit occuper 0,6 ha de `001`, l'eau 0,4 ha de `001` et 0,5 ha de `NA`.
Les doublons ne doivent ajouter aucune surface. Les six cellules des deux zones
et trois catégories doivent apparaître, même lorsqu'elles valent zéro.

In [3]:
result = cm.tabulate_area(zones, classes, **parameters)
assert isinstance(result, cm.AreaTabulationResult)
np.testing.assert_allclose(result.table['area'], [.6, .4, 0, 0, .5, 0])
np.testing.assert_allclose(result.table['percentage'], [60, 40, 0, 0, 50, 0])
assert result.matrix['zone_value'].tolist() == ['001', 'NA']
np.testing.assert_allclose(result.matrix.iloc[:, 3:], [[.6, .4, 0], [0, .5, 0]])
display(result.table)
display(result.matrix)
display(result.categories[['class_id', 'class_value', 'same_class_excess_area']])

,zone_id,zone_type,zone_value,class_id,class_type,class_value,area,percentage
0,z000001,string,001,c000001,string,Forêt,0.6,60.0
1,z000001,string,001,c000002,string,Eau,0.4,40.0
2,z000001,string,001,c000003,string,Agriculture,0.0,0.0
3,z000002,string,NA,c000001,string,Forêt,0.0,0.0
4,z000002,string,NA,c000002,string,Eau,0.5,50.0
5,z000002,string,NA,c000003,string,Agriculture,0.0,0.0


,zone_id,zone_type,zone_value,c000001,c000002,c000003
0,z000001,string,001,0.6,0.4,0.0
1,z000002,string,NA,0.0,0.5,0.0


,class_id,class_value,same_class_excess_area
0,c000001,Forêt,0.2
1,c000002,Eau,0.0
2,c000003,Agriculture,0.0


## 3. Couverture et fermeture des surfaces

La zone `NA` reste non couverte sur 0,5 ha. Le pourcentage d'une catégorie
utilise toute la superficie de la zone comme dénominateur. Le total du
territoire réuni est distinct d'une somme de zones qui se recouvriraient.

In [4]:
balance = result.zones.drop(columns='geometry')
np.testing.assert_allclose(balance['zone_area'], [1, 1])
np.testing.assert_allclose(balance['covered_area'], [1, .5])
np.testing.assert_allclose(balance['uncovered_area'], [0, .5])
np.testing.assert_allclose(balance['same_zone_excess_area'], [.5, 0])
np.testing.assert_allclose(balance['closure_error'], 0, atol=1e-12)
assert result.summary['totals']['domain']['zone_area'] == 2
assert result.summary['totals']['domain']['covered_area'] == 1.5
assert result.summary['totals']['domain']['uncovered_area'] == .5
assert set(result.overlaps['kind']) == {'same_class_overlap', 'same_zone_overlap'}
display(balance[['zone_value', 'zone_area', 'covered_area', 'uncovered_area', 'closure_error']])

,zone_value,zone_area,covered_area,uncovered_area,closure_error
0,001,1.0,1.0,0.0,0.0
1,NA,1.0,0.5,0.5,0.0


## 4. Recouvrement de catégories : décider explicitement

Le mode par défaut refuse les catégories qui se recouvrent. Le mode `report`
conserve chaque catégorie et produit les diagnostics. Aucun ordre de priorité
ne résout automatiquement une ambiguïté.

In [5]:
conflict = pd.concat([classes, gpd.GeoDataFrame({'occupation': ['Bâti']},
    geometry=[rectangle(50, 0, 80, 100)], crs=zones.crs)], ignore_index=True)
try:
    cm.tabulate_area(zones, conflict, **parameters)
except ValueError as error:
    assert 'overlap' in str(error).lower()
    print('Refus explicite du recouvrement :', str(error).splitlines()[0])
else:
    raise AssertionError('Le recouvrement aurait dû être refusé.')
reported = cm.tabulate_area(zones, conflict, **parameters, overlap_policy='report')
first = reported.zones.iloc[0]
np.testing.assert_allclose([first.covered_area, first.overlap_area, first.excess_area], [1, .3, .3])
assert np.isclose(first.summed_class_percentage, 130)
display(reported.zones[['zone_value', 'covered_area', 'overlap_area', 'excess_area', 'summed_class_percentage']])

Refus explicite du recouvrement : Distinct classes overlap with positive area inside the zones: c000001, c000004; use overlap_policy='report' to retain and audit both categories.


,zone_value,covered_area,overlap_area,excess_area,summed_class_percentage
0,001,1.0,0.3,0.3,130.0
1,NA,0.5,0.0,0.0,50.0


## 5. Un recouvrement triple ne se compte pas comme son emprise

Trois catégories recouvrent la zone selon les intervalles 0–80 m, 20–100 m
et 40–60 m. L'emprise touchée par au moins deux catégories vaut 0,6 ha.
La somme attribuée aux classes vaut 1,8 ha : l'excès de comptage vaut 0,8 ha.

In [6]:
triple = gpd.GeoDataFrame({'occupation': ['A', 'B', 'C']}, geometry=[
    rectangle(0, 0, 80, 100), rectangle(20, 0, 100, 100),
    rectangle(40, 0, 60, 100)], crs=zones.crs)
tripled = cm.tabulate_area(zones.iloc[:1], triple, **parameters, overlap_policy='report')
row = tripled.zones.iloc[0]
np.testing.assert_allclose([row.zone_area, row.summed_class_area, row.overlap_area, row.excess_area], [1, 1.8, .6, .8])
display(tripled.zones[['zone_value', 'zone_area', 'summed_class_area', 'overlap_area', 'excess_area']])

,zone_value,zone_area,summed_class_area,overlap_area,excess_area
0,001,1.0,1.8,0.6,0.8


## 6. Distinguer une catégorie absente d'une absence de couverture

Une emprise sans catégorie couvre 0,2 ha de la lacune de `NA`. Avec la règle
explicite `unclassified`, la lacune totale reste 0,5 ha : 0,2 ha sont non classés
et 0,3 ha restent sans couverture. Une catégorie connue conserve sa couverture.

In [7]:
with_null = pd.concat([classes, gpd.GeoDataFrame({'occupation': [None]},
    geometry=[rectangle(150, 0, 170, 100)], crs=zones.crs)], ignore_index=True)
missing = cm.tabulate_area(zones, with_null, **parameters, null_classes='unclassified')
row = missing.zones.iloc[1]
np.testing.assert_allclose([row.uncovered_area, row.unclassified_area, row.no_coverage_area], [.5, .2, .3])
assert set(missing.gaps['kind']) == {'unclassified', 'no_coverage'}
display(missing.gaps[['zone_value', 'kind', 'area', 'percentage']])

,zone_value,kind,area,percentage
0,NA,unclassified,0.2,20.0
1,NA,no_coverage,0.3,30.0


## 7. Publier et relire tous les produits

Les couches sont enregistrées en GeoPackage dans le SCR d'analyse. Les CSV
possèdent un schéma ; `001` et `NA` restent des libellés textuels. Les écritures
sont vérifiées avant publication du dossier. L'audit contient les empreintes
des fichiers vérifiés et les tables sans géométrie.

In [8]:
zp, cp = work / 'zones.gpkg', work / 'occupation.gpkg'
write_geopackage(zones, zp, layer='territoires')
write_geopackage(classes, cp, layer='occupation')
source_hashes = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in (zp, cp)}
file_parameters = dict(zones=str(zp), classes=str(cp), zones_layer='territoires',
    classes_layer='occupation', **parameters)
published = cm.execute_operation('vector.tabulate_area', file_parameters, work / 'surfaces')
saved = table_page(published['products']['table'])['frame']
assert saved['zone_value'].tolist() == ['001'] * 3 + ['NA'] * 3
np.testing.assert_allclose(saved['area'], result.table['area'])
assert cm.read_file(published['products']['zones']).crs == zones.crs
audit = json.loads(Path(published['products']['audit']).read_text(encoding='utf-8'))
for filename, proof in audit['verified_files'].items():
    assert hashlib.sha256((work / 'surfaces' / filename).read_bytes()).hexdigest() == proof['sha256']
display(saved)
print('Produits vérifiés :', ', '.join(published['products']))

,zone_id,zone_type,zone_value,class_id,class_type,class_value,area,percentage
0,z000001,string,001,c000001,string,Forêt,0.6,60.0
1,z000001,string,001,c000002,string,Eau,0.4,40.0
2,z000001,string,001,c000003,string,Agriculture,0.0,0.0
3,z000002,string,NA,c000001,string,Forêt,0.0,0.0
4,z000002,string,NA,c000002,string,Eau,0.5,50.0
5,z000002,string,NA,c000003,string,Agriculture,0.0,0.0


Produits vérifiés : primary, data, zones, gaps, overlaps, table, matrix, categories, audit, summary


## 8. Terminal et réutilisation dans un plan

La commande dédiée appelle le même opérateur. Un plan peut reprendre les
zones dissoutes ou les lacunes sans rechercher leurs chemins intermédiaires.

In [9]:
# Le processus enfant et le lecteur utilisent UTF-8, y compris sous Windows.
command = [sys.executable, '-X', 'utf8', '-m', 'cartomize', 'tabulate-area', str(zp), str(cp),
    str(work / 'terminal'), '--zone-field', 'territoire', '--class-field', 'occupation',
    '--zones-layer', 'territoires', '--classes-layer', 'occupation', '--area-unit', 'ha']
completed = subprocess.run(command, check=True, capture_output=True, text=True, encoding='utf-8')
cli_record = json.loads(completed.stdout)
np.testing.assert_allclose(table_page(cli_record['products']['table'])['frame']['area'], result.table['area'])
plan = cm.processing_plan([
    dict(id='surfaces', operation='vector.tabulate_area', parameters=file_parameters),
    dict(id='lacunes', operation='vector.area', parameters=dict(source='@surfaces:gaps', unit='ha')),
])
manifest = cm.run_plan(plan, work / 'plan')
plan_record = json.loads(Path(manifest).read_text(encoding='utf-8'))
assert np.isclose(cm.read_file(plan_record['products']['lacunes']['primary'])['area_ha'].sum(), .5)
print('Commande et plan : mêmes surfaces, lacunes réutilisables.')

Commande et plan : mêmes surfaces, lacunes réutilisables.


## 9. Annuler sans résultat partiel et contrôler les sources

L'annulation ci-dessous intervient pendant la publication, après le calcul.
Le dossier final doit rester absent et les empreintes des sources identiques.

In [10]:
event = Event()
def stop_during_publication(done, total):
    if done > 8000:
        event.set()
try:
    cm.execute_operation('vector.tabulate_area', file_parameters, work / 'annule',
                         cancel=event, progress=stop_during_publication)
except cm.ProcessingCancelled:
    pass
else:
    raise AssertionError('Annulation non propagée.')
assert not (work / 'annule').exists()
assert not list(work.glob('.cartomize-*'))
assert {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in (zp, cp)} == source_hashes
assert zones.geometry.to_wkb().tolist() == original_wkb[0]
assert classes.geometry.to_wkb().tolist() == original_wkb[1]
print('Annulation atomique et sources inchangées confirmées.')

Annulation atomique et sources inchangées confirmées.


## 10. Lire le résultat et expliquer les écarts

La carte de gauche montre les intersections classées ; celle de droite localise
la lacune. Le terrain est simulé. Une interprétation réelle doit relier la
projection, la date des données et la signification des classes à l'étude.

In [11]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
result.data.plot(column='class_value', categorical=True, legend=True, ax=axes[0],
                 cmap='Set2', edgecolor='white')
result.zones.boundary.plot(ax=axes[0], color='#243d32', linewidth=1.5)
result.zones.plot(ax=axes[1], color='#e5ece7', edgecolor='#243d32')
result.gaps.plot(ax=axes[1], color='#b3cbb9', edgecolor='#243d32', hatch='///')
for axis, title in zip(axes, ['Intersections — données simulées', 'Surface non couverte : 0,5 ha']):
    axis.set_title(title)
    axis.set_axis_off()
fig.savefig(work / 'surfaces.png', dpi=140, bbox_inches='tight')
plt.show()

## Exercices et réponses attendues

1. **Pourquoi la forêt ne mesure-t-elle pas 0,8 ha dans `001` ?** Les 0,2 ha
   dupliqués appartiennent à la même catégorie ; la réunion mesure 0,6 ha.
2. **Pourquoi l'eau ne couvre-t-elle que 50 % de `NA` ?** Le dénominateur est
   toute la zone de 1 ha, et non sa seule partie classée.
3. **Que signifient 0,6 ha de recouvrement et 0,8 ha d'excès dans le cas triple ?**
   L'emprise ambiguë est comptée une fois ; la partie couverte trois fois ajoute
   deux fois sa surface à l'excès.
4. **Peut-on additionner les zones si elles se recouvrent ?** Cette somme compte
   leurs parties communes plusieurs fois. Consulter le total `domain` pour la
   réunion du territoire, ou demander `zone_overlap_policy='error'`.

Les surfaces réelles de G09 sont vérifiées séparément dans le validateur
épinglé de Paris. [Guide et règles complètes](../../docs/VECTOR_TABULATION.md).